# Model Extension Workflow

Gendantic provides two functions to incrementally enhance your models:

1. **`extend_model_with_distributions()`**: Add statistical distributions to a basic model
2. **`extend_model_with_correlations()`**: Add correlations to a model with distributions

This allows you to:
- Start with a simple Pydantic model
- Let the LLM suggest appropriate distributions
- Let the LLM suggest realistic correlations

In [ ]:
from typing import Annotated

import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

from gendantic import (
    generate_synthetic_data,
    extend_model_with_distributions,
    extend_model_with_correlations,
    Normal,
    Uniform,
)

plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Start with a Basic Model

Define a simple Pydantic model without any distributions:

In [ ]:
class Employee(BaseModel):
    """Employee record for HR analytics."""
    name: str
    email: str
    age: int
    years_experience: int
    salary: float
    department: str
    performance_score: float = Field(ge=0, le=1)

print("Basic model fields:")
for name, field in Employee.model_fields.items():
    print(f"  {name}: {field.annotation}")

## Step 2: Add Distributions

Let the LLM suggest appropriate distributions based on field semantics:

In [ ]:
DistributedEmployee, dist_code = await extend_model_with_distributions(Employee)

print("Model with distributions:")
print("-" * 50)
print(dist_code)
print("-" * 50)

In [ ]:
# Generate data with the distributed model
employees = await generate_synthetic_data(DistributedEmployee, count=5, seed=42)

print("\nGenerated employees:")
for emp in employees:
    print(f"\n{emp.name} ({emp.email})")
    print(f"  Age: {emp.age} | Experience: {emp.years_experience} yrs")
    print(f"  Salary: £{emp.salary:,.0f} | Dept: {emp.department}")
    print(f"  Performance: {emp.performance_score:.2f}")

## Step 3: Add Correlations

Now let the LLM suggest realistic correlations between the distribution fields:

In [ ]:
CorrelatedEmployee, corr_code = await extend_model_with_correlations(DistributedEmployee)

print("Model with correlations:")
print("-" * 50)
print(corr_code)
print("-" * 50)

In [ ]:
# Generate data with correlations
correlated_employees = await generate_synthetic_data(CorrelatedEmployee, count=200, seed=42)

# Verify correlations
ages = [e.age for e in correlated_employees]
exp = [e.years_experience for e in correlated_employees]
sal = [e.salary for e in correlated_employees]
perf = [e.performance_score for e in correlated_employees]

print("Correlation matrix:")
print(f"  Age-Experience: {np.corrcoef(ages, exp)[0,1]:.2f}")
print(f"  Experience-Salary: {np.corrcoef(exp, sal)[0,1]:.2f}")
print(f"  Performance-Salary: {np.corrcoef(perf, sal)[0,1]:.2f}")

# Visualize the correlations
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].scatter(ages, exp, alpha=0.4, s=20)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Years Experience')
axes[0].set_title(f'Age vs Experience\n(r={np.corrcoef(ages, exp)[0,1]:.2f})')

axes[1].scatter(exp, sal, alpha=0.4, s=20, color='green')
axes[1].set_xlabel('Years Experience')
axes[1].set_ylabel('Salary (£)')
axes[1].set_title(f'Experience vs Salary\n(r={np.corrcoef(exp, sal)[0,1]:.2f})')

axes[2].scatter(perf, sal, alpha=0.4, s=20, color='orange')
axes[2].set_xlabel('Performance Score')
axes[2].set_ylabel('Salary (£)')
axes[2].set_title(f'Performance vs Salary\n(r={np.corrcoef(perf, sal)[0,1]:.2f})')

plt.tight_layout()
plt.show()

In [ ]:
# Notice how values make sense together
print("\nSample correlated employees:")
for emp in correlated_employees[:5]:
    print(f"  Age {emp.age}, {emp.years_experience} yrs exp, £{emp.salary:,.0f}, perf {emp.performance_score:.2f}")

## Full Workflow Example: Sales Data

In [ ]:
# Step 1: Basic model
class SalesRecord(BaseModel):
    """Sales transaction record."""
    customer_name: str
    product_name: str
    deal_value: float
    units_sold: int
    discount_pct: float = Field(ge=0, le=0.5)
    days_to_close: int
    region: str

print("Step 1: Basic model defined")

In [ ]:
# Step 2: Add distributions
DistributedSales, sales_dist_code = await extend_model_with_distributions(SalesRecord)

print("Step 2: Distributions added")
print(sales_dist_code)

In [ ]:
# Step 3: Add correlations
CorrelatedSales, sales_corr_code = await extend_model_with_correlations(DistributedSales)

print("Step 3: Correlations added")
print(sales_corr_code)

In [ ]:
# Generate final data
sales = await generate_synthetic_data(CorrelatedSales, count=5, seed=42)

print("\nGenerated sales records:")
for s in sales:
    print(f"\n{s.customer_name} - {s.product_name}")
    print(f"  Value: £{s.deal_value:,.0f} | Units: {s.units_sold} | Discount: {s.discount_pct:.1%}")
    print(f"  Region: {s.region} | Days to close: {s.days_to_close}")

## Starting from a Model with Distributions

If you already have a model with distributions, you can go directly to adding correlations:

In [ ]:
class Project(BaseModel):
    """Project tracking model."""
    project_name: str
    team_size: Annotated[int, Uniform(min=2, max=20)]
    budget: Annotated[float, Normal(mean=100000, std=50000)]
    duration_weeks: Annotated[int, Uniform(min=4, max=52)]
    complexity_score: Annotated[float, Uniform(min=0, max=1)]

# Skip extend_model_with_distributions, go straight to correlations
CorrelatedProject, project_code = await extend_model_with_correlations(Project)

print("Model with correlations:")
print(project_code)

In [ ]:
projects = await generate_synthetic_data(CorrelatedProject, count=100, seed=42)

# Extract data
team_sizes = [p.team_size for p in projects]
budgets = [p.budget for p in projects]
durations = [p.duration_weeks for p in projects]
complexity = [p.complexity_score for p in projects]

# Visualize correlations
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].scatter(team_sizes, budgets, alpha=0.5, s=30)
axes[0].set_xlabel('Team Size')
axes[0].set_ylabel('Budget (£)')
axes[0].set_title(f'Team Size vs Budget\n(r={np.corrcoef(team_sizes, budgets)[0,1]:.2f})')

axes[1].scatter(durations, complexity, alpha=0.5, s=30, color='green')
axes[1].set_xlabel('Duration (weeks)')
axes[1].set_ylabel('Complexity Score')
axes[1].set_title(f'Duration vs Complexity\n(r={np.corrcoef(durations, complexity)[0,1]:.2f})')

axes[2].scatter(team_sizes, complexity, alpha=0.5, s=30, color='orange')
axes[2].set_xlabel('Team Size')
axes[2].set_ylabel('Complexity Score')
axes[2].set_title(f'Team Size vs Complexity\n(r={np.corrcoef(team_sizes, complexity)[0,1]:.2f})')

plt.tight_layout()
plt.show()

print("\nSample projects:")
for p in projects[:3]:
    print(f"  {p.project_name}: Team {p.team_size}, £{p.budget:,.0f}, {p.duration_weeks}wks, complexity {p.complexity_score:.2f}")

## Workflow Summary

```
Basic Model
    │
    ▼  extend_model_with_distributions()
Model with Distributions
    │
    ▼  extend_model_with_correlations()
Model with Distributions + Correlations
    │
    ▼  generate_synthetic_data()
Realistic Synthetic Data
```

Benefits:
- **Incremental**: Add complexity step by step
- **Inspectable**: See what the LLM suggests at each step
- **Controllable**: Start with manual distributions, only use LLM for correlations
- **Reproducible**: Save the generated code for future use

## Saving Generated Models

The generated code can be saved and reused:

In [ ]:
# Save the final model code to a file
final_code = f'''
from typing import Annotated
from pydantic import BaseModel, Field
from gendantic import Normal, Uniform, Categorical, Correlations

{sales_corr_code}
'''

print("Code ready to save to a .py file:")
print(final_code)